#### #1. Data Ingestion & Safe Date Standardization Staging

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import IntegerType

# Extract data from Bronze staging layer
bronze_df = spark.table("workspace.bronze.crm_sales_details")

# Use try_to_date to safely handle '0' or malformed inputs by converting them to NULL
standardized_df = bronze_df
date_cols = ["sls_order_dt", "sls_ship_dt", "sls_due_dt"]
for date_col in date_cols:
    standardized_df = standardized_df.withColumn(
        date_col, 
        F.try_to_date(F.col(date_col).cast("string"), "yyyyMMdd")
    )

#### # 2. Business Rules & Metrics Cleansing Logic

In [0]:
# Isolate calculation expressions for variance resolution and price derivations
transformed_df = (
    standardized_df
    # Fix invalid sales values (handles NULL, zero, negative, or mathematical mismatch)
    .withColumn(
        "sls_sales",
        F.when(
            F.col("sls_sales").isNull() | 
            (F.col("sls_sales") <= 0) | 
            (F.col("sls_sales") != (F.col("sls_quantity") * F.abs(F.col("sls_price")))),
            F.col("sls_quantity") * F.abs(F.col("sls_price"))
        ).otherwise(F.col("sls_sales"))
    )
    # Derive missing/invalid price safely (handling division by zero via inline check)
    .withColumn(
        "sls_price",
        F.when(
            F.col("sls_price").isNull() | (F.col("sls_price") <= 0),
            F.col("sls_sales") / F.when(F.col("sls_quantity") == 0, F.lit(None)).otherwise(F.col("sls_quantity"))
        ).otherwise(F.col("sls_price"))
    )
    # Add metadata operational column matching SQL's DEFAULT GETDATE()
    .withColumn("dwh_create_date", F.current_timestamp())
)

#### # 3. Final Schema Formatting, Renaming, and Target Storage

In [0]:
# Grouping DDL casting and structural organization together
final_df = transformed_df.select(
    F.col("sls_ord_num").cast("string"),
    F.col("sls_prd_key").cast("string"),
    F.col("sls_cust_id").cast(IntegerType()),
    F.col("sls_order_dt"),
    F.col("sls_ship_dt"),
    F.col("sls_due_dt"),
    F.col("sls_sales").cast(IntegerType()),
    F.col("sls_quantity").cast(IntegerType()),
    F.col("sls_price").cast(IntegerType()),
    F.col("dwh_create_date")
)

# Reference-aligned mapping to convert technical abbreviations to clear business terms
RENAME_MAP = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "shipping_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}

renamed_df = final_df
for old_name, new_name in RENAME_MAP.items():
    renamed_df = renamed_df.withColumnRenamed(old_name, new_name)

# Write output schema directly to Silver Delta layer (handles truncation automatically via overwrite)
renamed_df.write \
    .mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.crm_sales")

# Display sample preview rows interactively without variable name errors
renamed_df.display()